# Time Series Forecasting: Sales Prediction

**Category:** 12-Time-Series  
**Description:** Build forecasting models with AI-assisted analysis  
**Data:** ~/lab/data/CSVs/Warehouse_and_Retail_Sales.csv

---

## What You'll Learn

1. Time series decomposition
2. Trend and seasonality analysis
3. ARIMA and exponential smoothing
4. Prophet forecasting
5. AI-assisted pattern interpretation

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q pandas numpy matplotlib seaborn statsmodels prophet scikit-learn

# Model aliases
CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
GPT = "openai-chat:gpt-5"
GEMINI = "gemini:gemini-2.5-flash"

MODEL = CLAUDE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
DATA_DIR = Path.home() / 'lab' / 'data' / 'CSVs'

---

# Part 1: Data Loading & Preparation

---

In [ ]:
# Load sales data
df = pd.read_csv(DATA_DIR / 'Warehouse_and_Retail_Sales.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
# Create date column and aggregate to monthly
df['date'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str) + '-01')

# Aggregate monthly sales
monthly = df.groupby('date').agg({
    'retail_sales': 'sum',
    'warehouse_sales': 'sum'
}).reset_index()

monthly = monthly.sort_values('date').set_index('date')
monthly.index = pd.DatetimeIndex(monthly.index, freq='MS')

print(f"Monthly data shape: {monthly.shape}")
print(f"Date range: {monthly.index.min()} to {monthly.index.max()}")
monthly.head()

In [ ]:
# Plot the time series
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(monthly.index, monthly['retail_sales'] / 1e6, 'b-', linewidth=2)
axes[0].set_title('Monthly Retail Sales', fontsize=14)
axes[0].set_ylabel('Sales (Millions $)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(monthly.index, monthly['warehouse_sales'] / 1e6, 'g-', linewidth=2)
axes[1].set_title('Monthly Warehouse Sales', fontsize=14)
axes[1].set_ylabel('Sales (Millions $)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# Part 2: Time Series Decomposition

---

In [ ]:
# Focus on retail sales for forecasting
ts = monthly['retail_sales'].copy()

# Decompose the time series
decomposition = seasonal_decompose(ts, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))

decomposition.observed.plot(ax=axes[0], title='Observed')
decomposition.trend.plot(ax=axes[1], title='Trend')
decomposition.seasonal.plot(ax=axes[2], title='Seasonal')
decomposition.resid.plot(ax=axes[3], title='Residual')

plt.tight_layout()
plt.show()

In [ ]:
# Seasonal patterns by month
monthly_avg = ts.groupby(ts.index.month).mean()

plt.figure(figsize=(10, 5))
plt.bar(range(1, 13), monthly_avg.values / 1e6, color='steelblue')
plt.xlabel('Month')
plt.ylabel('Average Sales (Millions $)')
plt.title('Average Sales by Month')
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.show()

## 2.1 AI Pattern Analysis

In [ ]:
%%ai $MODEL
I've decomposed a retail sales time series and found:
- Clear upward trend over time
- Strong seasonal pattern with peaks in certain months
- Some residual volatility

Based on typical retail patterns:
1. What might explain the seasonal peaks?
2. What external factors could affect the trend?
3. What forecasting approach would you recommend?
4. What risks should we consider when forecasting retail sales?

---

# Part 3: Stationarity Testing

---

In [ ]:
def test_stationarity(series, title=''):
    """Perform Augmented Dickey-Fuller test for stationarity."""
    result = adfuller(series.dropna())
    
    print(f"\n{title}")
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print(f"Critical Values:")
    for key, value in result[4].items():
        print(f"  {key}: {value:.4f}")
    
    if result[1] <= 0.05:
        print("=> Series is STATIONARY (reject null hypothesis)")
    else:
        print("=> Series is NON-STATIONARY (fail to reject null hypothesis)")
    
    return result[1] <= 0.05

# Test original series
is_stationary = test_stationarity(ts, "Original Series")

In [ ]:
# If not stationary, difference the series
if not is_stationary:
    ts_diff = ts.diff().dropna()
    test_stationarity(ts_diff, "First Difference")
    
    # Plot differenced series
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    
    axes[0].plot(ts, 'b-')
    axes[0].set_title('Original Series')
    
    axes[1].plot(ts_diff, 'g-')
    axes[1].set_title('First Difference')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ACF and PACF plots
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plot_acf(ts.diff().dropna(), lags=24, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF)')

plot_pacf(ts.diff().dropna(), lags=24, ax=axes[1])
axes[1].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.show()

---

# Part 4: ARIMA Forecasting

---

In [ ]:
# Split data into train and test
train_size = int(len(ts) * 0.8)
train, test = ts[:train_size], ts[train_size:]

print(f"Training: {train.index.min()} to {train.index.max()} ({len(train)} months)")
print(f"Testing: {test.index.min()} to {test.index.max()} ({len(test)} months)")

In [ ]:
# Fit ARIMA model
# Parameters: (p, d, q) - based on ACF/PACF analysis
model_arima = ARIMA(train, order=(2, 1, 2))
fitted_arima = model_arima.fit()

print(fitted_arima.summary())

In [ ]:
# Forecast
forecast_arima = fitted_arima.forecast(steps=len(test))

# Calculate metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae_arima = mean_absolute_error(test, forecast_arima)
rmse_arima = np.sqrt(mean_squared_error(test, forecast_arima))
mape_arima = np.mean(np.abs((test - forecast_arima) / test)) * 100

print(f"ARIMA Forecast Metrics:")
print(f"  MAE:  ${mae_arima:,.0f}")
print(f"  RMSE: ${rmse_arima:,.0f}")
print(f"  MAPE: {mape_arima:.2f}%")

In [ ]:
# Plot ARIMA forecast
plt.figure(figsize=(14, 6))

plt.plot(train.index, train / 1e6, 'b-', label='Training')
plt.plot(test.index, test / 1e6, 'g-', label='Actual')
plt.plot(test.index, forecast_arima / 1e6, 'r--', label='ARIMA Forecast')

plt.fill_between(test.index, 
                 (forecast_arima - 1.96 * rmse_arima) / 1e6,
                 (forecast_arima + 1.96 * rmse_arima) / 1e6,
                 color='red', alpha=0.2, label='95% CI')

plt.title('ARIMA Forecast vs Actual', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Sales (Millions $)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

# Part 5: Exponential Smoothing

---

In [ ]:
# Fit Holt-Winters Exponential Smoothing
model_hw = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='mul',
    seasonal_periods=12
)
fitted_hw = model_hw.fit()

# Forecast
forecast_hw = fitted_hw.forecast(steps=len(test))

# Metrics
mae_hw = mean_absolute_error(test, forecast_hw)
rmse_hw = np.sqrt(mean_squared_error(test, forecast_hw))
mape_hw = np.mean(np.abs((test - forecast_hw) / test)) * 100

print(f"Holt-Winters Forecast Metrics:")
print(f"  MAE:  ${mae_hw:,.0f}")
print(f"  RMSE: ${rmse_hw:,.0f}")
print(f"  MAPE: {mape_hw:.2f}%")

In [ ]:
# Plot comparison
plt.figure(figsize=(14, 6))

plt.plot(train.index, train / 1e6, 'b-', label='Training', alpha=0.7)
plt.plot(test.index, test / 1e6, 'k-', label='Actual', linewidth=2)
plt.plot(test.index, forecast_arima / 1e6, 'r--', label=f'ARIMA (MAPE: {mape_arima:.1f}%)')
plt.plot(test.index, forecast_hw / 1e6, 'g--', label=f'Holt-Winters (MAPE: {mape_hw:.1f}%)')

plt.title('Model Comparison: ARIMA vs Holt-Winters', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Sales (Millions $)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

# Part 6: Prophet Forecasting

---

In [ ]:
from prophet import Prophet

# Prepare data for Prophet (requires 'ds' and 'y' columns)
prophet_train = pd.DataFrame({
    'ds': train.index,
    'y': train.values
})

prophet_test = pd.DataFrame({
    'ds': test.index,
    'y': test.values
})

# Fit Prophet model
model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)
model_prophet.fit(prophet_train)

In [ ]:
# Make predictions
future = model_prophet.make_future_dataframe(periods=len(test), freq='MS')
forecast_prophet = model_prophet.predict(future)

# Extract test period predictions
prophet_pred = forecast_prophet[forecast_prophet['ds'].isin(test.index)]['yhat'].values

# Metrics
mae_prophet = mean_absolute_error(test, prophet_pred)
rmse_prophet = np.sqrt(mean_squared_error(test, prophet_pred))
mape_prophet = np.mean(np.abs((test.values - prophet_pred) / test.values)) * 100

print(f"Prophet Forecast Metrics:")
print(f"  MAE:  ${mae_prophet:,.0f}")
print(f"  RMSE: ${rmse_prophet:,.0f}")
print(f"  MAPE: {mape_prophet:.2f}%")

In [ ]:
# Prophet's built-in plot
fig1 = model_prophet.plot(forecast_prophet)
plt.title('Prophet Forecast')
plt.show()

# Component plots
fig2 = model_prophet.plot_components(forecast_prophet)
plt.show()

---

# Part 7: Model Comparison Summary

---

In [ ]:
# Compare all models
comparison = pd.DataFrame({
    'Model': ['ARIMA', 'Holt-Winters', 'Prophet'],
    'MAE ($)': [mae_arima, mae_hw, mae_prophet],
    'RMSE ($)': [rmse_arima, rmse_hw, rmse_prophet],
    'MAPE (%)': [mape_arima, mape_hw, mape_prophet]
})

comparison = comparison.round(2)
print("\nModel Comparison:")
print(comparison.to_string(index=False))

best_model = comparison.loc[comparison['MAPE (%)'].idxmin(), 'Model']
print(f"\nBest Model: {best_model}")

In [ ]:
# Visualization of comparison
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(comparison))
width = 0.25

ax.bar(x - width, comparison['MAE ($)'] / 1e6, width, label='MAE (M$)')
ax.bar(x, comparison['RMSE ($)'] / 1e6, width, label='RMSE (M$)')
ax.bar(x + width, comparison['MAPE (%)'], width, label='MAPE (%)')

ax.set_ylabel('Error Metric')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'])
ax.legend()

plt.tight_layout()
plt.show()

---

# Part 8: AI-Powered Insights

---

In [ ]:
%%ai $MODEL
I compared three time series forecasting models for retail sales:

1. ARIMA(2,1,2) - Traditional statistical model
2. Holt-Winters Exponential Smoothing - Handles trend and seasonality
3. Facebook Prophet - Modern, handles holidays and changepoints

The data shows:
- Strong monthly seasonality
- Overall upward trend
- Some volatility in residuals

Please provide:
1. Interpretation of which model is likely best and why
2. Business recommendations based on forecasting
3. What additional features could improve forecasts (holidays, promotions, etc.)
4. How to communicate forecast uncertainty to stakeholders

---

# Part 9: Future Forecast

---

In [ ]:
# Retrain on full data and forecast 12 months ahead
model_full = Prophet(yearly_seasonality=True)
prophet_full = pd.DataFrame({'ds': ts.index, 'y': ts.values})
model_full.fit(prophet_full)

# Forecast 12 months
future_dates = model_full.make_future_dataframe(periods=12, freq='MS')
future_forecast = model_full.predict(future_dates)

# Plot
fig = model_full.plot(future_forecast)
plt.title('12-Month Sales Forecast')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.show()

# Show forecast values
print("\n12-Month Forecast:")
future_only = future_forecast[future_forecast['ds'] > ts.index.max()][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
future_only['yhat_formatted'] = (future_only['yhat'] / 1e6).round(2).astype(str) + 'M'
print(future_only.to_string(index=False))

---

## Summary

This notebook demonstrated:

1. **Time Series Decomposition** - Trend, seasonality, residuals
2. **Stationarity Testing** - ADF test, differencing
3. **ARIMA Modeling** - Classical statistical approach
4. **Exponential Smoothing** - Holt-Winters with seasonality
5. **Prophet Forecasting** - Modern, scalable forecasting
6. **Model Comparison** - MAE, RMSE, MAPE metrics
7. **AI Insights** - Business interpretation
8. **Future Forecasting** - 12-month predictions

---

**Next:** Try anomaly detection in `12-Time-Series/02-anomaly-detection.ipynb`